This notebook serves as a proof of concept for searching spikes in place names to identify events.
The last cell [`places_1577.ipynb`](../notebooks/places_1577.ipynb) uncovers Scotland as occuring significantly in Edward I.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import xml.etree.ElementTree as ET
import re
import textwrap

repo_root = Path.cwd().resolve()
if not (repo_root / "1577_files").is_dir():
    repo_root = repo_root.parent

# Locate the (Edward) XML file
edward_path = repo_root / "1577_files" / "regnal_files" / "09_EdwardFirst.xml"

TEI = "{http://www.tei-c.org/ns/1.0}"

root = ET.parse(edward_path).getroot()

# Find Edward I's reign section
edward_entry = root.find(f".//{TEI}div[@type='reign'][@n='Edward I']")

if edward_entry is None:
    raise ValueError("Could not find Edward I's reign section.")

# Extract and normalize the historical text
edward_xml_words = re.sub(r"\s+", " ", " ".join(edward_entry.itertext())).strip()

print(textwrap.fill(edward_xml_words[:1000], width=150))

In [ ]:
# Split into list of words
edward_words = edward_xml_words.split()

# Find occurrences of "Scotland" and "Scotlande"
scotland_pattern = re.compile(r"(?<!\w)Scotland(?:e)?(?!\w)", re.IGNORECASE)

scotland_positions = [
    match.start() for match in scotland_pattern.finditer(edward_xml_words)
]

# Convert character positions to word positions
word_positions = [len(edward_xml_words[:pos].split()) for pos in scotland_positions]

print(f"Found {len(word_positions)} mentions of Scotland/Scotlande.")

# Count mentions in consecutive 500-word sections
window_size = 500
number_of_windows = (len(edward_words) + window_size - 1) // window_size

scotland_counts = (
    pd.Series(word_positions)
    .floordiv(window_size)
    .value_counts()
    .sort_index()
    .reindex(range(number_of_windows), fill_value=0)
)

# Extract marginal years and map each one to its word position in the text
year_pattern = re.compile(r"(?<!\d)(?:12(?:7[2-9]|[89]\d)|130[0-7])(?!\d)")
year_markers = []
search_from = 0

for note in edward_entry.findall(f".//{TEI}note[@place='margin']"):
    note_text = re.sub(r"\s+", " ", " ".join(note.itertext())).strip()
    matches = list(year_pattern.finditer(note_text))
    if not matches:
        continue

    note_offset = edward_xml_words.find(note_text, search_from)
    if note_offset < 0:
        raise ValueError(f"Could not locate marginal note in extracted text: {note_text!r}")

    for match in matches:
        word_position = len(edward_xml_words[:note_offset + match.start()].split())
        year_markers.append((int(match.group()), word_position))

    search_from = note_offset + len(note_text)

display(scotland_counts.sort_values(ascending=False).head(10))

# Plot mentions across the text
plt.figure(figsize=(16, 5))

plt.bar(
    scotland_counts.index * window_size,
    scotland_counts.values,
    width=window_size * 0.9,
    align="edge",
)

ax = plt.gca()
max_count = scotland_counts.max()
for i, (year, word_position) in enumerate(year_markers):
    ax.axvline(word_position, color="darkred", alpha=0.25, linewidth=0.8)
    ax.text(
        word_position,
        max_count + 1 + (i % 4) * 2.5,
        str(year),
        rotation=90,
        ha="center",
        va="bottom",
        color="darkred",
        fontsize=8,
    )
ax.set_ylim(0, max_count + 13)

plt.xlabel("Word in Edward I's text")
plt.ylabel("Mentions per 500-word section")
plt.title("Scotland/Scotlande Mentions Across Edward I's Reign")
plt.xticks(ticks=range(0, len(edward_words) + 1, 2000), rotation=45)
plt.tight_layout()
plt.show()

We see a spike between words 12000-16000

In [ ]:
# Load the people CSV
people_df = pd.read_csv(repo_root / "name_extraction" / "entity_names" / "holinshed_people.csv")

# Extract the text slice
eleven_seventeen = " ".join(edward_words[11000:17000])

# Count occurrences of each person's name
def count_person_mentions(row):
    # Match whole names, ignoring capitalization
    pattern = r"(?<!\w)" + re.escape(row["name"]) + r"(?!\w)"

    return len(re.findall(pattern, eleven_seventeen, flags=re.IGNORECASE))

people_df["count"] = people_df.apply(count_person_mentions, axis=1)

# Show people who appear, ordered by count
people_in_slice = (
    people_df[people_df["count"] > 0]
    [["name", "count"]]
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(people_in_slice.head(10))



In [ ]:
# Reproduce previous cell with top names to see where they are mentioned
slice_start = 11000
slice_end = 17000
top_people = people_in_slice.head(8)["name"].tolist()

person_positions = {}
for name in top_people:
    pattern = r"(?<!\w)" + re.escape(name) + r"(?!\w)"
    person_positions[name] = [
        slice_start + len(eleven_seventeen[:m.start()].split())
        for m in re.finditer(pattern, eleven_seventeen, flags=re.IGNORECASE)
    ]

fig, (ax_bar, ax_people) = plt.subplots(
    2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]}
)

# Top panel: identical to the Cell 3 bar chart (same data, bins, and styling)
ax_bar.bar(
    scotland_counts.index * window_size,
    scotland_counts.values,
    width=window_size * 0.9,
    align="edge",
)
ax_bar.set_ylabel("Mentions per 500-word section")
ax_bar.set_title("Scotland/Scotlande Mentions Across Edward I's Reign")

# Bottom panel: one row (a "swimlane") per person, on the same word axis as above
colors = plt.cm.tab10.colors
for i, (name, positions) in enumerate(person_positions.items()):
    ax_people.eventplot(
        positions,
        lineoffsets=i,
        linelengths=0.8,
        linewidths=2,
        colors=colors[i % len(colors)],
    )

ax_people.set_yticks(range(len(person_positions)))
ax_people.set_yticklabels(list(person_positions.keys()))
ax_people.set_ylim(-0.5, len(person_positions) - 0.5)
ax_people.invert_yaxis()
ax_people.set_xlabel("Word in Edward I's text")
ax_people.set_xticks(range(slice_start, slice_end + 1, 500))
ax_people.tick_params(axis="x", rotation=45)

# Zoom both panels (they share the x-axis) into the slice the names were searched in
ax_people.set_xlim(slice_start, slice_end)

fig.tight_layout()
plt.show()


In [ ]:
# Quickly examine where a few people appear
section = " ".join(edward_words[11750:12000])
print(textwrap.fill(section, width=150))